In [1]:
# Import the libraries
from autogen import ConversableAgent, GroupChat, GroupChatManager, UserProxyAgent

In [2]:
from getpass import getpass
import os 

os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API Key: ")

In [3]:
# configuration for LLM
config_list = {
    "config_list": [{"model": "gpt-4o-mini", "temperature": 0.1}]
}

In [4]:
user_agent = UserProxyAgent(
    name="user_financial_inputs",
    llm_config=False,
    human_input_mode="NEVER",
    code_execution_config=False,
)

portfolio_analytics_agent = ConversableAgent(
    name="portfolio_analytics_agent",
    system_message="""
    You are a portfolio analysis agent. Summarize the user's holdings, risk profile,
    time horizon, liquidity needs, and diversification. Classify the portfolio as
    either Growth or Value, and explain the classification in 3-5 bullets.
    """,
    llm_config=config_list,
    human_input_mode="NEVER",
)

investment_type_agent = ConversableAgent(
    name="investment_recommendation_agent",
    system_message="""
    You are an investment recommendation coordinator. Use the portfolio analysis
    to request nested recommendations from the growth and value specialists, then
    compare their suggestions and identify which recommendation style best fits
    the user's stated goals and risk tolerance.
    """,
    llm_config=config_list,
    human_input_mode="NEVER",
)

growth_investment_agent = ConversableAgent(
    name="growth_investment_agent",
    system_message="""
    You are a growth investment specialist. Recommend higher-growth portfolio
    adjustments for users with suitable risk tolerance and longer time horizons.
    Include rationale, expected benefits, and key risks.
    """,
    llm_config=config_list,
    human_input_mode="NEVER",
)

value_investment_agent = ConversableAgent(
    name="value_investment_agent",
    system_message="""
    You are a value investment specialist. Recommend stable, long-term portfolio
    adjustments focused on quality, valuation discipline, income, and downside
    protection. Include rationale, expected benefits, and key risks.
    """,
    llm_config=config_list,
    human_input_mode="NEVER",
)

advisor_investment_agent = ConversableAgent(
    name="advisor_investment_agent",
    system_message="""
    You are a financial advisor agent. Compile a personalized final report using
    the portfolio analysis and investment recommendations. Include: portfolio
    snapshot, Growth/Value classification, recommended allocation changes, risks,
    and next steps. Do not present the output as guaranteed financial advice.
    """,
    llm_config=config_list,
    human_input_mode="NEVER",
)

In [5]:
nested_chats = [
    {
        "recipient": growth_investment_agent,
        "message": "Review the portfolio analysis and provide high-growth investment recommendations tailored to the user's goals and risk tolerance.",
        "max_turns": 1,
        "summary_method": "last_msg",
    },
    {
        "recipient": value_investment_agent,
        "message": "Review the portfolio analysis and provide stable, long-term value investment recommendations tailored to the user's goals and risk tolerance.",
        "max_turns": 1,
        "summary_method": "last_msg",
    },
]

investment_type_agent.register_nested_chats(
    nested_chats,
    trigger=lambda sender: sender not in [growth_investment_agent, value_investment_agent],
)

In [6]:
def state_transition(last_speaker, groupchat):
    _ = groupchat

    if last_speaker is user_agent:
        return portfolio_analytics_agent
    elif last_speaker is portfolio_analytics_agent:
        return investment_type_agent
    elif last_speaker is investment_type_agent:
        return advisor_investment_agent
    elif last_speaker is advisor_investment_agent:
        return None
    return None

In [7]:
group_chat = GroupChat(
    agents=[portfolio_analytics_agent, investment_type_agent, advisor_investment_agent],
    messages=[],
    max_round=10,
    speaker_selection_method=state_transition,
    select_speaker_auto_llm_config=config_list
)

group_chat_manager = GroupChatManager(
    groupchat=group_chat,
)

In [ ]:
from IPython.display import Markdown, display

portfolio_request = """
User profile:
- Age: 34
- Investment horizon: 12-15 years
- Risk tolerance: moderately aggressive
- Goal: long-term wealth creation with some downside protection
- Monthly investment capacity: $2,000

Current portfolio:
- 45% large-cap technology stocks
- 20% broad-market index funds
- 15% international equity funds
- 10% bonds
- 5% REITs
- 5% cash

Please analyze the portfolio, classify it as Growth or Value, compare suitable
investment recommendations, and prepare a final personalized report.
"""

chat_result = user_agent.initiate_chat(
    group_chat_manager,
    message=portfolio_request,
    summary_method="last_msg",
)

final_report = next(
    (
        message["content"]
        for message in reversed(group_chat.messages)
        if message.get("name") == advisor_investment_agent.name
    ),
    chat_result.summary,
)

display(Markdown(final_report))

user_financial_inputs (to chat_manager):


User profile:
- Age: 34
- Investment horizon: 12-15 years
- Risk tolerance: moderately aggressive
- Goal: long-term wealth creation with some downside protection
- Monthly investment capacity: $2,000

Current portfolio:
- 45% large-cap technology stocks
- 20% broad-market index funds
- 15% international equity funds
- 10% bonds
- 5% REITs
- 5% cash

Please analyze the portfolio, classify it as Growth or Value, compare suitable
investment recommendations, and prepare a final personalized report.


--------------------------------------------------------------------------------

Next speaker: portfolio_analytics_agent

portfolio_analytics_agent (to chat_manager):

### Portfolio Summary

**Holdings:**
- **Large-Cap Technology Stocks:** 45%
- **Broad-Market Index Funds:** 20%
- **International Equity Funds:** 15%
- **Bonds:** 10%
- **REITs:** 5%
- **Cash:** 5%

**Risk Profile:**
- **Moderately Aggressive:** The portfolio is heavily weighted toward


User profile:
- Age: 34
- Investment horizon: 12-15 years
- Risk tolerance: moderately aggressive
- Goal: long-term wealth creation with some downside protection
- Monthly investment capacity: $2,000

Current portfolio:
- 45% large-cap technology stocks
- 20% broad-market index funds
- 15% international equity funds
- 10% bonds
- 5% REITs
- 5% cash

Please analyze the portfolio, classify it as Growth or Value, compare suitable
investment recommendations, and prepare a final personalized report.


In [16]:
final_report = next(
    (
        message["content"]
        for message in reversed(group_chat.messages)
        if message.get("name") == advisor_investment_agent.name
    ),
    chat_result.summary,
)

display(Markdown(final_report))

### Personalized Final Report

**User Profile:**
- **Age:** 34
- **Investment Horizon:** 12-15 years
- **Risk Tolerance:** Moderately Aggressive
- **Monthly Investment Capacity:** $2,000

---

### Portfolio Snapshot

**Current Portfolio Allocation:**
- **Large-Cap Technology Stocks:** 45%
- **Broad-Market Index Funds:** 20%
- **International Equity Funds:** 15%
- **Bonds:** 10%
- **REITs:** 5%
- **Cash:** 5%

### Growth/Value Classification

**Classification:** Growth

**Justification:**
- The portfolio is heavily weighted towards equities, particularly in the technology sector, which is known for higher volatility and potential returns.
- A significant portion (65%) is allocated to equities, indicating a growth-oriented strategy.
- The limited fixed income allocation (10%) suggests a lower emphasis on value-oriented investments, which typically have higher allocations to stable, dividend-paying stocks.

---

### Recommended Allocation Changes

1. **Reduce Large-Cap Technology Stocks:**
   - **New Allocation:** Decrease from 45% to 35%.
   - **Rationale:** Mitigate risk by diversifying away from a single sector.

2. **Increase Dividend Aristocrats:**
   - **New Allocation:** Increase from 0% to 10%.
   - **Rationale:** Provide regular income and stability during market downturns.

3. **Enhance Fixed Income Exposure:**
   - **New Allocation:** Increase from 10% to 15%.
   - **Rationale:** Provide more stability and downside protection.

4. **Increase REITs:**
   - **New Allocation:** Increase from 5% to 10%.
   - **Rationale:** Hedge against inflation and provide income through dividends.

5. **Explore Defensive Sectors (Utilities and Healthcare):**
   - **New Allocation:** Introduce a 5% allocation to each sector.
   - **Rationale:** Stability and consistent cash flows during economic downturns.

6. **Maintain Broad-Market Index Funds and International Equity Funds:**
   - **Allocation:** Keep at 20% and 15%, respectively.
   - **Rationale:** Ensure continued exposure to market growth and diversification.

---

### Risks

- **Market Volatility:** High exposure to equities, particularly in technology, can lead to significant fluctuations in portfolio value.
- **Sector Concentration:** Heavy reliance on technology stocks increases risk if that sector underperforms.
- **Interest Rate Risk:** Increasing bond allocation may expose the portfolio to interest rate fluctuations, impacting bond prices.
- **Economic Downturns:** While defensive sectors provide some protection, overall market conditions can still affect portfolio performance.

---

### Next Steps

1. **Implement Recommended Changes:**
   - Gradually adjust the portfolio allocations as outlined above. Consider dollar-cost averaging into new investments to mitigate market timing risks.

2. **Regular Portfolio Review:**
   - Schedule quarterly reviews to assess performance and make necessary adjustments based on market conditions and personal financial goals.

3. **Stay Informed:**
   - Keep abreast of market trends, economic indicators, and changes in your personal financial situation that may necessitate further adjustments.

4. **Consult a Financial Advisor:**
   - Consider working with a financial advisor to ensure that your investment strategy aligns with your long-term goals and risk tolerance.

---

This personalized report aims to provide a comprehensive overview of your current investment strategy while offering actionable recommendations to enhance your portfolio's stability and growth potential. Always remember that investing involves risks, and it's essential to make informed decisions based on your unique financial situation.